# 01주 · 통계적 질문·자료 정리

공학통계학 CE202
슬라이드의 개념·숫자·풀이 순서와 연결한 **새 병행 실습**. 원본 Colab 복원본이 아닌 신규 자료

**목표**: 자료·확률·분포·추정·검정의 원리 이해와 통계적 해석 심화

학습 주제: 교과목 소개 / 자료 정리

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE202-statistics-for-engineers/week01/CE202_W01.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE202-statistics-for-engineers/week01)

질문·가정 확인 → 손계산·예측 → 계산·시각화 → 한 조건 변경 → 통계적 해석·한계 설명

셀 실행: Shift+Enter. 아래 코드의 None은 직접 채울 항목. 원본과 변경 사본의 결과를 구분

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE202-statistics-for-engineers/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

## 0. 실행 환경

필요한 패키지의 최소 버전 확인. 기본 코드와 보조 함수는 이번 주 실습에 필요한 도구. 세부 구현 암기 불필요

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.16", "matplotlib": "3.10", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math, tempfile, base64, zipfile, io
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
WORK = Path(tempfile.mkdtemp(prefix="ce202-"))
rng = np.random.default_rng(2027)

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 수치 자료와 최소 표본수 확인 필요")
    return x

def probabilities(values):
    p = vector(values)
    if np.any(p < 0) or not np.isclose(p.sum(), 1):
        raise ValueError("확률의 비음수성·합계 1 확인 필요")
    return p

def mean_ci(values, confidence=0.95):
    x = vector(values, 2)
    if not 0 < confidence < 1:
        raise ValueError("신뢰수준은 0과 1 사이")
    se = x.std(ddof=1) / np.sqrt(len(x))
    q = stats.t.ppf((1 + confidence) / 2, df=len(x)-1)
    return {"n": len(x), "mean": x.mean(), "se": se,
            "df": len(x)-1, "low": x.mean()-q*se, "high": x.mean()+q*se}

def wilson_interval(successes, n, confidence=0.95):
    if not isinstance(n, (int, np.integer)) or n <= 0 or not 0 <= successes <= n:
        raise ValueError("0 ≤ 성공수 ≤ 양의 정수 표본수 확인 필요")
    if int(successes) != successes or not 0 < confidence < 1:
        raise ValueError("정수 성공수·신뢰수준 확인 필요")
    z = stats.norm.ppf((1+confidence)/2)
    phat = successes / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z / den * np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))
    return center-half, center+half

def anova_summary(groups):
    arrays = [vector(g, 2) for g in groups]
    if len(arrays) < 2:
        raise ValueError("두 집단 이상 필요")
    n = np.array([len(g) for g in arrays]); means = np.array([g.mean() for g in arrays])
    grand = np.concatenate(arrays).mean()
    ss_between = np.sum(n*(means-grand)**2)
    ss_within = sum(np.sum((g-g.mean())**2) for g in arrays)
    df_between = len(arrays)-1; df_within = n.sum()-len(arrays)
    if ss_within <= 0:
        raise ValueError("집단 내 변동이 0이면 일반적인 F 계산 불가")
    F = (ss_between/df_between) / (ss_within/df_within)
    return {"ss_between": ss_between, "ss_within": ss_within,
            "df_between": df_between, "df_within": df_within,
            "F": F, "p": stats.f.sf(F, df_between, df_within)}

def regression(x, y):
    x = vector(x, 3); y = vector(y, 3)
    if len(x) != len(y) or np.ptp(x) == 0:
        raise ValueError("같은 길이·서로 다른 x 값 확인 필요")
    fit = stats.linregress(x, y)
    fitted = fit.intercept + fit.slope*x
    residual = y-fitted
    sse = np.sum(residual**2); sst = np.sum((y-y.mean())**2)
    if sst <= 0:
        raise ValueError("y 변동이 0이면 일반적인 R² 해석 불가")
    return {"slope": fit.slope, "intercept": fit.intercept, "r2": 1-sse/sst,
            "sse": sse, "mse_residual": sse/(len(x)-2), "mse_prediction": sse/len(x),
            "p": fit.pvalue, "stderr": fit.stderr, "fitted": fitted, "residual": residual}


## S01. 통계적 질문을 분석 명세로 바꾸기

**슬라이드 연결**: CE202_W01a_Intro.pptx p8, 9, 11, 13, 15

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 색상 번호에 평균을 계산해도 의미가 있는가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
analysis_spec = {"question": "스마트폰 색상 선호 분포", "unit": "응답자 1명", "variable": "선호 색상", "scale": "명목", "population": "원본 예제의 조사 대상 소비자", "sample_n": 500}
print(pd.Series(analysis_spec))

**조건 변경**: 같은 명세에서 모집단과 표본을 구분. 모집단을 임의로 대한민국 전체로 확장하지 않는지 확인

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**선택 심화 · AI 검토**: 관측단위·변수척도·표본수·모집단·원하는 표를 명시한 요약 요청

**결과 검토**: 표본의 선호 분포와 모집단의 선호 분포를 구분

예측·손계산·변경 결과·개념 설명·적용 한계 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## S02. 원본 색상 선호 도수표

**슬라이드 연결**: CE202_W01b_Data.pptx p7, 8

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 화이트의 상대도수와 전체 합계 예측

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
colours = ["화이트", "블랙", "블루", "레드", "그레이", "핑크"]
counts = np.array([180,60,75,45,50,90])
frequency = pd.DataFrame({"색상": colours, "도수": counts, "상대도수": counts/counts.sum()})
print(frequency)
assert counts.sum() == 500 and np.isclose(frequency["상대도수"].sum(),1)
plt.bar(["white","black","blue","red","grey","pink"], counts); plt.ylabel("Count"); plt.show()

**조건 변경**: 화이트 도수만 180→200으로 변경한 사본에서 분모도 바뀌는지 확인

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**선택 심화 · AI 검토**: 6개 범주·원본 도수·총수 500을 제시하고 빈도표·막대그래프 요청

**결과 검토**: 범주 누락·중복과 분모를 먼저 검토

예측·손계산·변경 결과·개념 설명·적용 한계 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## S03. 원본 KBO 연봉 자료 정렬

**슬라이드 연결**: CE202_W01b_Data.pptx p5

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 중복값을 제거하면 원본 자료수가 유지되는가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
salary = np.array([12,10,20,8,23,12,12.5,13,20,15,13,12,12.5,7,6.5,20,8,25,6.5,7])
print("내림차순:", np.sort(salary)[::-1]); print("자료수:",len(salary))
assert len(salary)==20 and salary.max()==25

**조건 변경**: 정렬과 중복 제거의 결과를 별도 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**선택 심화 · AI 검토**: 2023 KBO 예제의 익명 연봉값 20개·단위 억원·중복 보존 조건 명시

**결과 검토**: 선수 이름을 추정하거나 현재 연봉으로 해석하지 않는지 확인

예측·손계산·변경 결과·개념 설명·적용 한계 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## 주차 확인

명목형 색상자료의 적절한 요약과 화이트 비율 기록

입력 항목:

- `method`: 명목형 색상자료의 요약 방법. 도수표는 frequency · 문자열
- `white_fraction`: 전체 응답자 중 화이트 비율 (0~1) · 수치
- `numeric_mean_meaningful`: 색상 코드 번호의 산술평균이 의미 있는가? · Python `True`/`False`

공개 확인용 항목. 결과값만 복사한 경우 학습 완료로 판단 불가. 분석 가정·자료 출처·추론 범위는 아래 기록과 함께 검토

In [ ]:
answers = {'method': None, 'white_fraction': None, 'numeric_mean_meaningful': None}


In [ ]:
def check_answers(values):
    expected = {'method': 'frequency', 'white_fraction': 0.36, 'numeric_mean_meaningful': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음

In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
